In [1]:
from config import Config
from DataManager import DataManager
from DataPreprocessor import Preprocessor
from FeatureSelection import FeatureSelector
from ModelManager import ModelManager
from ModelEvaluation import Evaluator

from DataExplorer import Explorer #new


from sklearn.model_selection import train_test_split

import pandas as pd
import numpy as np

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
cfg = Config()
dm = DataManager(cfg)
preprocessor = Preprocessor(cfg)

gt_df = dm.load_ground_truth()
wl = dm.load_wavelengths()

In [4]:
X_train, valid_train= preprocessor.process_dataset(dm, image_ids=dm.get_data_indices(cfg.TRAIN_DATA_PATH), file_path=cfg.TRAIN_DATA_PATH, include_higher_derivs=True)
y_train = gt_df.loc[valid_train,:].values

print(f'X_train shape {X_train.shape}, y shape {y_train.shape}')

processed 100/1732 images...
processed 200/1732 images...
processed 300/1732 images...
processed 400/1732 images...
processed 500/1732 images...
processed 600/1732 images...
processed 700/1732 images...
processed 800/1732 images...
processed 900/1732 images...
processed 1000/1732 images...
processed 1100/1732 images...
processed 1200/1732 images...
processed 1300/1732 images...
processed 1400/1732 images...
processed 1500/1732 images...
processed 1600/1732 images...
processed 1700/1732 images...
X_train shape (1732, 450), y shape (1732, 4)


In [10]:
X_test, _= preprocessor.process_dataset(dm, image_ids=dm.get_data_indices(cfg.TEST_DATA_PATH), file_path=cfg.TEST_DATA_PATH, include_higher_derivs=True)

print(f'X_test shape {X_test.shape}')

processed 100/1154 images...
processed 200/1154 images...
processed 300/1154 images...
processed 400/1154 images...
processed 500/1154 images...
processed 600/1154 images...
processed 700/1154 images...
processed 800/1154 images...
processed 900/1154 images...
processed 1000/1154 images...
processed 1100/1154 images...
X_test shape (1154, 450)


In [ ]:
#X_train, X_test, y_train, y_test = train_test_split(X_train, y_train, test_size=0.2, random_state=cfg.RANDOM_SEED)

In [ ]:
image_data = dm.load_image(1700)#1700 image - takie samo jak wcześniej
Explorer.compare_masked_vs_not_masked(image_data, 50 )


In [ ]:
Explorer.plot_single_image_spectrum(image_data, wl)

In [ ]:
#Calculating and plotting first, second and third order derivatives of avg spectral reflectance curve
Explorer.plot_image_derivatives(image_data, wl)

In [ ]:
#Calculating correlation between bands of the example image
Explorer.plot_band_correlation_matrix(image_data)

In [5]:
# setting up RFE Selector class, fitting and transforming train set
selector = FeatureSelector(cfg, n_features=50, step=10) 

X_train_rfe = selector.fit_transform(X_train, y_train)


Fitting estimator with 450 features.
Fitting estimator with 440 features.
Fitting estimator with 430 features.
Fitting estimator with 420 features.
Fitting estimator with 410 features.
Fitting estimator with 400 features.
Fitting estimator with 390 features.
Fitting estimator with 380 features.
Fitting estimator with 370 features.
Fitting estimator with 360 features.
Fitting estimator with 350 features.
Fitting estimator with 340 features.
Fitting estimator with 330 features.
Fitting estimator with 320 features.
Fitting estimator with 310 features.
Fitting estimator with 300 features.
Fitting estimator with 290 features.
Fitting estimator with 280 features.
Fitting estimator with 270 features.
Fitting estimator with 260 features.
Fitting estimator with 250 features.
Fitting estimator with 240 features.
Fitting estimator with 230 features.
Fitting estimator with 220 features.
Fitting estimator with 210 features.
Fitting estimator with 200 features.
Fitting estimator with 190 features.
F

In [ ]:
#saving selector state to the pkl file
selector.save()

Successfully saved selector to: ../Submissions\Selectors\selector.pkl!


In [18]:
#tests with a new FeatureSelector instance
new_selector = FeatureSelector(cfg)
new_selector.load()

new_selector.get_indices() == selector.get_indices()

array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True])

In [ ]:
#transforming test set
selector.support_.shape
X_test_rfe = selector.transform(X_test)

In [ ]:
#setting up model manager with model type being RandomForestRegressor
models = ModelManager(cfg, model_type='rf')

#tuning and fitting model via SearchCV
models.tune_fit(X_train_rfe, y_train, verbose=True)

#getting model ensembler predictions
y_pred = models.predict(X_test_rfe)


Fitting 5 folds for each of 80 candidates, totalling 400 fits
Best params for [P] parameter: {'n_estimators': 400, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'log2', 'max_depth': 20}

Fitting 5 folds for each of 80 candidates, totalling 400 fits
Best params for [K] parameter: {'n_estimators': 500, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 20}

Fitting 5 folds for each of 80 candidates, totalling 400 fits
Best params for [Mg] parameter: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 20}

Fitting 5 folds for each of 80 candidates, totalling 400 fits
Best params for [pH] parameter: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 20}



In [ ]:
# Model evaluation, hyperview score available only when splitting data ino train_test_split, for real hyperview score sending submission file on the server is required
evaluator = Evaluator(cfg)
y_baseline = ModelManager.get_baseline_predictions(X_train=X_train_rfe, y_train=y_train,X_test=X_test_rfe)
#evaluator.calculate_hyperview_score(y_test, y_pred, y_baseline)

P              0.996034
K              0.900700
Mg             0.899135
pH             0.850717
FINAL_SCORE    0.911647
dtype: float64

In [ ]:
#saving submission file and pickldumping best rf model into a pkl file
dm.save_submission(y_pred, filename='submission_rf')
models.save_models(filename='best_rf_model.pkl')

#===================================================

Succesfully saved submission file to: ../Submissions\submission_rf.csv
Successfully saved models to: ../Submissions\Models\best_rf_model.pkl!


In [29]:
# new model manager instance for testing
models_rf = ModelManager(cfg)
models_rf.load_models('best_rf_model.pkl')
y_pred_new = models_rf.predict(X_test_rfe)


In [6]:
#test wyboru pliku potem do wyrzucenua

from Users import Farmer
cfg = Config()
farmer = Farmer(cfg, user_id=1) #

print("odpalamy okno")

# metoda ktora pozwala w konie windowsa wybrac obraz
farmer.load_data_from_desktop()

if farmer.data is not None:
    print("\nWczytano obraz.")
    print(f"Wymiary danych: {farmer.data.shape}")
else:
    print("\nAnulowano wybór lub wystąpił błąd.")


odpalamy okno

Anulowano wybór lub wystąpił błąd.
